In [ ]:
# Colab setup: mount Google Drive and configure project paths.
# Run this cell first on Colab. If running locally, skip this cell.
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Aini/ml-assignment/Team-Assignment-2/binus-ai-2026sem3-assignment2-group04"
MODEL_DIR   = f"{PROJECT_DIR}/models"
TUNER_DIR   = f"{PROJECT_DIR}/keras_tuner"
!mkdir -p "$MODEL_DIR"
!mkdir -p "$TUNER_DIR"

import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Project dir:", PROJECT_DIR)
print("Model dir:  ", MODEL_DIR)
print("Tuner dir:  ", TUNER_DIR)

# Tugas Kelompok 2 — Phase 1C: Hyperparameter Tuning dengan Keras Tuner

**Tujuan:**
Mencari set hyperparameter yang lebih baik untuk arsitektur baseline VGG-style dari `01_baseline_EN.ipynb` (test acc 0,8735), menggunakan **Keras Tuner Hyperband**. Search space menargetkan knob yang paling mungkin menggerakkan jarum: learning rate, dropout per blok, L2 weight decay, lebar dense head, dan pilihan optimizer.

**Strategi pencarian — Hyperband (Li et al. 2017):**
Hyperband adalah allocator gaya bandit yang adaptif. Kandidat murah berjalan beberapa epoch; yang menjanjikan mendapat compute lebih, yang lemah dipangkas lebih awal. Dengan `max_epochs=20` dan `factor=3` algoritma menjelajahi ~30-40 konfigurasi berbeda sambil menghabiskan sebagian besar budget pada top performer — jauh lebih efisien dari grid atau random search pada cost yang sama.

**Mengapa setelah Phase 1B (transfer learning):**
Kami sudah tahu transfer learning (0,8993) mengungguli baseline dari nol. Tujuan HP tuning *bukan* mengalahkan transfer learning — tidak bisa, keduanya dibatasi oleh skala input 32×32 pada arsitektur ini. Tujuannya adalah:
1. **Mengetahui seberapa banyak headroom baseline berasal dari pilihan HP vs. arsitektur.** Jika model tuned mendapat ≥1 pp di atas hand-tuned, HP manual meninggalkan kemenangan mudah; jika datar, hand-tuning kami sudah dekat optimal untuk arsitektur ini.
2. **Menghasilkan model ketiga untuk bab perbandingan laporan** (`baseline_v1` → `baseline_tuned_v2` → `transfer_mobilenet_v1`), yang merupakan pusat narasi optimisasi.

**Catatan untuk versi ID:** untuk menghemat waktu (search asli ~1,5-2 jam + retrain ~30 menit), notebook ini melewati Hyperband search dan pelatihan ulang 50 epoch dengan memuat artifact yang dihasilkan oleh `02_hyperparameter_tuning_EN.ipynb`:
- `best_hp.json` — hyperparameter pemenang
- `baseline_tuned_v2.keras` — model akhir terlatih
- `baseline_tuned_v2_history.pkl` — riwayat 50 epoch retrain

Section 8-12 tetap dieksekusi pada model yang dimuat untuk menghasilkan output Indonesia.

### Bagian 0: Setup dan Reproduktibilitas

Kami menggunakan kembali `SEED = 42` dari baseline sehingga pembagian train/validation identik — search HP dan baseline keduanya dievaluasi pada val set yang identik bit-by-bit, yang diperlukan untuk perbandingan yang adil.

Hyperband sendiri memperkenalkan randomness tambahan (trial mana yang di-sample, yang dipromosikan) yang tidak di-seed. Jadi jika menjalankan ulang notebook ini dari nol, HP yang dipilih kemungkinan akan sedikit berbeda. Direktori `keras_tuner/` cache di Drive memastikan resumability dalam satu search.

In [ ]:
SEED = 42
BATCH_SIZE = 64

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, os

print(f"Versi TensorFlow:    {tf.__version__}")
print(f"GPU tersedia:        {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Random seed:         {SEED}")
print(f"Batch size:          {BATCH_SIZE}")

### Bagian 1: Memuat CIFAR-10 (Hugging Face)

Sumber Hugging Face yang sama dengan notebook baseline, dengan alasan yang sama — mirror U Toronto yang ditunjukkan helper bawaan Keras tidak stabil. Mirror HF dihosting di HF Hub CDN.

In [ ]:
try:
    import datasets
except ImportError:
    !pip install -q datasets
    import datasets

print("Mengunduh CIFAR-10 via Hugging Face datasets...")
hf_dataset = datasets.load_dataset("cifar10")
hf_dataset.set_format("numpy")

train_images = np.array(hf_dataset['train']['img'])
train_labels = np.array(hf_dataset['train']['label']).reshape(-1, 1)
test_images  = np.array(hf_dataset['test']['img'])
test_labels  = np.array(hf_dataset['test']['label']).reshape(-1, 1)

class_names = ['pesawat', 'mobil', 'burung', 'kucing', 'rusa',
               'anjing', 'katak', 'kuda', 'kapal', 'truk']

print("Data berhasil dimuat.")
print(f"Sampel training: {len(train_images)}, sampel test: {len(test_images)}")

### Bagian 2: Preprocessing dan Stratified Train/Validation Split

Identik dengan pipeline baseline (normalisasi → split dengan `SEED=42` → one-hot encoding). Menggunakan ulang val set yang sama memastikan model tuned-HP dievaluasi pada contoh yang persis sama dengan baseline, sehingga perbedaan akurasi dapat dikaitkan dengan pilihan HP, bukan keberuntungan val set.

In [ ]:
X_train_full = train_images.astype('float32') / 255.0
X_test       = test_images.astype('float32') / 255.0
y_train_full = train_labels.flatten()
y_test_int   = test_labels.flatten()

X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_train_full, y_train_full,
    test_size=0.1, random_state=SEED, stratify=y_train_full,
)

y_train = to_categorical(y_train_int, num_classes=10)
y_val   = to_categorical(y_val_int,   num_classes=10)
y_test  = to_categorical(y_test_int,  num_classes=10)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

### Bagian 3: Pipeline Data Augmentation

Pipeline `tf.data` yang sama dengan baseline: horizontal flip ringan + rotasi kecil + zoom kecil hanya untuk training; validation dan test dataset diteruskan tanpa augmentation. Augmentation tetap (tidak di-tune) sehingga perubahan akurasi yang dapat dikaitkan ke HP tidak tercampur dengan kekuatan augmentation.

In [ ]:
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name='augmenter')

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10000, seed=SEED)
            .batch(BATCH_SIZE)
            .map(lambda x, y: (augmenter(x, training=True), y),
                 num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f"train_ds: {len(X_train)} sampel (dengan augmentation)")
print(f"val_ds:   {len(X_val)} sampel")
print(f"test_ds:  {len(X_test)} sampel")

### Bagian 4: HyperModel — Arsitektur yang Dapat Dicari

Arsitektur sama dengan CNN VGG-style 3-blok dari baseline. Kami mengekspos hyperparameter berikut ke Keras Tuner:

| Hyperparameter | Range | Sampling |
|---|---|---|
| `learning_rate` | 1e-4 – 1e-2 | log-uniform |
| `dropout_block1` | 0,1 – 0,4 | step 0,1 |
| `dropout_block2` | 0,2 – 0,5 | step 0,1 |
| `dropout_block3` | 0,3 – 0,5 | step 0,1 |
| `weight_decay` (L2) | 1e-5 – 1e-3 | log-uniform |
| `dense_units` | 64 / 128 / 256 | choice |
| `optimizer` | adam / adamw / sgd | choice |

7 dimensi — cukup besar sehingga grid search tidak feasible. Hyperband melakukan sampling secara efisien di dalam product space ini.

Dropout classifier-head (0,5 sebelum Dense terakhir) **tidak** di-tune — head adalah titik risiko overfitting tertinggi dan standar literatur 0,5 dari Srivastava dkk. (2014) sudah teruji.

In [ ]:
try:
    import keras_tuner as kt
except ImportError:
    !pip install -q keras-tuner
    import keras_tuner as kt

print(f"Versi Keras Tuner: {kt.__version__}")


def build_model(hp):
    """HyperModel untuk Keras Tuner. Arsitektur VGG-style yang sama dengan
    baseline; hanya knob yang terdaftar yang diekspos ke search."""
    weight_decay = hp.Float('weight_decay', 1e-5, 1e-3, sampling='log')
    dropout1 = hp.Float('dropout_block1', 0.1, 0.4, step=0.1)
    dropout2 = hp.Float('dropout_block2', 0.2, 0.5, step=0.1)
    dropout3 = hp.Float('dropout_block3', 0.3, 0.5, step=0.1)
    dense_units = hp.Choice('dense_units', [64, 128, 256])
    optimizer_name = hp.Choice('optimizer', ['adam', 'adamw', 'sgd'])
    learning_rate = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')

    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Blok 1: 32 filter
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout1),

        # Blok 2: 64 filter
        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout2),

        # Blok 3: 128 filter
        layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout3),

        # Classifier head
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ])

    if optimizer_name == 'adam':
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'adamw':
        opt = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    else:  # sgd
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

### Bagian 5: Hyperband Search

> **Catatan ID:** versi ini melewati Hyperband search dan langsung memuat `best_hp.json` yang dihasilkan oleh `02_hyperparameter_tuning_EN.ipynb`. Logika search asli tetap di-preserve sebagai cabang fallback (untuk dokumentasi reproduksi).

Pengaturan Hyperband (untuk referensi):
- `max_epochs=20` — cap setiap trial pada 20 epoch (search horizon).
- `factor=3` — rasio successive halving.
- `objective='val_accuracy'` — metrik yang menggerakkan keputusan promosi.

Total trial: sekitar 30-40 (Hyperband memutuskan otomatis dari rumus). Wall-clock asli (run EN): ~1 jam.

In [ ]:
best_hp_path = f"{MODEL_DIR}/best_hp.json"
HP_LOADED = os.path.exists(best_hp_path)

if HP_LOADED:
    print(f"Memuat hyperparameter dari {best_hp_path}")
    print("(Melewati Hyperband search — hasil dihasilkan oleh 02_hyperparameter_tuning_EN.ipynb)")
    with open(best_hp_path) as f:
        best_hp_values = json.load(f)
    print("\nHyperband search dilewati. Lihat run EN untuk log search lengkap.")
else:
    # Logika fallback — jalankan search asli
    tuner = kt.Hyperband(
        build_model,
        objective='val_accuracy',
        max_epochs=20,
        factor=3,
        directory=TUNER_DIR,
        project_name='cifar10_baseline_hpsearch',
        overwrite=False,
        seed=SEED,
    )
    tuner.search_space_summary()
    print("\n" + "=" * 65)
    print("Memulai Hyperband search (~1,5-2 jam pada T4 GPU)")
    print("=" * 65)
    tuner.search(
        train_ds, validation_data=val_ds,
        callbacks=[
            callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                    restore_best_weights=True, verbose=0),
        ],
        verbose=1,
    )
    print("\nSearch selesai.")
    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_hp_values = best_hp.values

### Bagian 6: Hyperparameter Terbaik

Kami mengekstrak hyperparameter trial teratas dan mempertahankannya ke `models/best_hp.json` untuk laporan dan reproduktibilitas. Section berikutnya melatih ulang model fresh dengan HP ini untuk full 50-epoch budget (Hyperband hanya melihat trial 20-epoch, jadi model akhir bisa memeras lebih banyak dari HP yang sama).

In [ ]:
print("Hyperparameter terbaik dari run Hyperband (EN):")
print("-" * 65)
for k, v in best_hp_values.items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v:.5g}")
    else:
        print(f"  {k:20s}: {v}")

# Persistensi (no-op jika sudah ada)
if not HP_LOADED:
    with open(f"{MODEL_DIR}/best_hp.json", "w") as f:
        json.dump(best_hp_values, f, indent=2)
    print(f"\nDisimpan ke {MODEL_DIR}/best_hp.json")
else:
    print(f"\nAlready persisted: {best_hp_path}")

### Bagian 7: Latih Ulang Model Terbaik untuk Full 50 Epoch

> **Catatan ID:** versi ini memuat `baseline_tuned_v2.keras` yang sudah dihasilkan run EN, alih-alih melatih ulang. Logika fit asli tetap dipertahankan sebagai cabang fallback.

In [ ]:
model_path = f"{MODEL_DIR}/baseline_tuned_v2.keras"
history_path = f"{MODEL_DIR}/baseline_tuned_v2_history.pkl"

MODEL_LOADED = HP_LOADED and os.path.exists(model_path) and os.path.exists(history_path)

class _StubHistory:
    def __init__(self, d): self.history = d

if MODEL_LOADED:
    print(f"Memuat model dari {model_path}")
    print("(Melewati pelatihan ulang 50 epoch — model dihasilkan oleh 02_hyperparameter_tuning_EN.ipynb)")
    best_model = tf.keras.models.load_model(model_path)
    with open(history_path, "rb") as f:
        h = pickle.load(f)
    history = _StubHistory(h)
    print(f"\nRetrained {len(history.history.get('loss', []))} epoch.")
    if history.history.get('val_accuracy'):
        print(f"Validation accuracy terbaik: {max(history.history['val_accuracy']):.4f}")
else:
    # Logika fallback — train ulang
    best_hp_obj = kt.HyperParameters()
    for k, v in best_hp_values.items():
        best_hp_obj.values[k] = v
    best_model = build_model(best_hp_obj)
    best_model.summary()
    print("\nMelatih ulang kandidat HP terbaik untuk full 50 epoch...")
    print("-" * 65)
    callback_list = [
        callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3,
                                    factor=0.5, min_lr=1e-6, verbose=1),
    ]
    history = best_model.fit(
        train_ds, validation_data=val_ds,
        epochs=50, callbacks=callback_list, verbose=1,
    )
    print(f"\nPelatihan akhir selesai setelah {len(history.history['loss'])} epoch.")
    print(f"Validation loss terbaik:     {min(history.history['val_loss']):.4f}")
    print(f"Validation accuracy terbaik: {max(history.history['val_accuracy']):.4f}")

### Bagian 8: Riwayat Pelatihan

Plot kurva accuracy dan loss dengan cara yang sama dengan notebook baseline, sehingga keduanya dapat dibandingkan secara visual berdampingan di laporan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Tuned Baseline — Akurasi Model')
axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Tuned Baseline — Loss Model')
axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Bagian 9: Evaluasi pada Test Set

Protokol sama dengan notebook baseline: agregat test loss / accuracy, lalu per-kelas precision, recall, dan F1.

In [ ]:
print("Mengevaluasi pada test set...")
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

y_pred_proba = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test_int

print("Laporan klasifikasi per kelas:")
print("-" * 65)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

### Bagian 10: Confusion Matrix

Tumpang tindih visual dengan confusion matrix baseline memberitahu kita apakah HP tuning menggeser *jenis* kesalahan yang dibuat model (misal mengurangi kerancuan kucing↔anjing secara spesifik) atau hanya mengurangi kesalahan secara seragam. Yang terakhir lebih khas untuk HP tuning yang tidak mengubah arsitektur.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Jumlah'})
plt.xlabel('Label Prediksi')
plt.ylabel('Label Sebenarnya')
plt.title('Confusion Matrix - Tuned Baseline (Hyperband HP) - Test Set CIFAR-10')
plt.tight_layout()
plt.show()

### Bagian 11: Misclassification dengan Confidence Tertinggi

9 gambar test di mana model tuned paling yakin salah. Dibandingkan dengan misclassification confidence-tertinggi baseline, nilai confidence absolut sering turun sedikit setelah tuning — efek samping dari regularisasi yang lebih ter-kalibrasi.

In [ ]:
misclassified_mask = (y_pred != y_true)
misclassified_indices = np.where(misclassified_mask)[0]

predicted_confidence = y_pred_proba[misclassified_indices, y_pred[misclassified_indices]]
top9_local_idx = np.argsort(predicted_confidence)[-9:][::-1]
top9_global_idx = misclassified_indices[top9_local_idx]

plt.figure(figsize=(12, 12))
for i, idx in enumerate(top9_global_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx])
    true_class = class_names[y_true[idx]]
    pred_class = class_names[y_pred[idx]]
    confidence = y_pred_proba[idx, y_pred[idx]]
    plt.title(f"{true_class} -> {pred_class}\n(confidence: {confidence:.1%})", fontsize=11)
    plt.axis('off')

plt.suptitle('9 Misclassification dengan Confidence Tertinggi (Tuned Baseline)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

### Bagian 12: Tabel Perbandingan Tiga Model

Ini adalah perbandingan kanonik yang akan diandalkan oleh Bab 6 laporan. Kami hard-code angka yang sebelumnya diketahui untuk `baseline_v1` (0,8735) dan `transfer_mobilenet_v1` (0,8993), dan menghitung angka `baseline_tuned_v2` notebook ini secara live.

In [ ]:
import pandas as pd

trainable_count = sum(int(tf.size(v).numpy()) for v in best_model.trainable_weights)

comparison = pd.DataFrame([
    {
        'Model': 'baseline_v1 (hand-tuned)',
        'Test Accuracy': 0.8735,
        'Trainable Params': '~570 K',
        'Notes': 'VGG-style, HP manual dari 01_baseline_EN.ipynb',
    },
    {
        'Model': 'baseline_tuned_v2 (Hyperband)',
        'Test Accuracy': round(test_acc, 4),
        'Trainable Params': f'{trainable_count:,}',
        'Notes': 'Arsitektur sama, HP dicari Hyperband',
    },
    {
        'Model': 'transfer_mobilenet_v1',
        'Test Accuracy': 0.8993,
        'Trainable Params': '~1,52 M',
        'Notes': 'MobileNetV2 ImageNet pretrained, fine-tuned pada 96x96',
    },
])

print("Perbandingan tiga model (tabel kanonik untuk Bab 6 laporan):")
print("=" * 75)
print(comparison.to_string(index=False))
print("=" * 75)
print(f"\nGap (tuned vs hand-tuned baseline):   {round((test_acc - 0.8735) * 100, 2):+} pp")
print(f"Gap (tuned vs transfer learning):     {round((test_acc - 0.8993) * 100, 2):+} pp")

### Bagian 13: Simpan Model Terlatih dan Riwayat

Untuk versi ID, cell di bawah men-skip penyimpanan jika model sudah ada di Drive (dihasilkan oleh run EN). Cabang fallback (else) tetap melakukan save.

- `baseline_tuned_v2.keras` — arsip lengkap Keras v3
- `baseline_tuned_v2_history.pkl` — dict `history.history` (untuk plotting di laporan)
- `best_hp.json` — sudah disimpan di §6 (artifact search-time)

In [ ]:
if MODEL_LOADED:
    print("Model sudah disimpan oleh 02_hyperparameter_tuning_EN.ipynb. Tidak perlu menyimpan ulang.")
    for f in ['baseline_tuned_v2.keras', 'baseline_tuned_v2_history.pkl', 'best_hp.json']:
        p = f"{MODEL_DIR}/{f}"
        if os.path.exists(p):
            print(f"  ✓ {p}  ({os.path.getsize(p) / 1e6:.2f} MB)")
        else:
            print(f"  ✗ {p}  (tidak ditemukan)")
else:
    model_path = f"{MODEL_DIR}/baseline_tuned_v2.keras"
    best_model.save(model_path)
    print(f"Model disimpan: {model_path}  ({os.path.getsize(model_path) / 1e6:.1f} MB)")
    history_path = f"{MODEL_DIR}/baseline_tuned_v2_history.pkl"
    with open(history_path, "wb") as f:
        pickle.dump(history.history, f)
    print(f"History disimpan: {history_path}")